# RAG poisoning detection: FINAL runner (finish and save everything)
**Run the cells top to bottom, one at a time.** Wait for each to finish (the play button comes back) and check for a red error before the next.

**State of the project:** both experiments are already finished and saved on your Google Drive:
- `nq_qwen3b_contriever`: main run, 3 poison passages per question (attack, features, A2, evaluate, filter all done)
- `nq_qwen3b_n1`: ablation, 1 poison passage per question (attack, features, evaluate, filter done; no A2 by design)

So tomorrow you only need **Steps 1 to 4 (CPU is enough, about 10 minutes)**. Step 5 is optional. The appendix at the bottom is only for rebuilding everything from scratch under NEW run names.

**Never re-run `attack` for the two finished runs.** The code now refuses to, to protect your results.

## Step 1. Connect Drive and set paths
Use any runtime for Steps 1 to 4 (CPU is fine).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["RAGDET_ROOT"] = "/content/drive/MyDrive/ragdet"        # durable files (Drive)
os.environ["RAGDET_DATA"] = "/content/data"                        # big raw corpus (not needed here)
os.environ["HF_HOME"]     = "/content/hf_cache"                    # model cache on local disk
os.environ["PYTHONPATH"]  = "/content/rag-poison-detect/src"
print("paths set")

## Step 2. Get the latest code from GitHub, then run the tests
Expected: **`33 passed, 1 skipped`**. If the number is lower, the newest code did not reach GitHub; stop and ask.

In [ ]:
%cd /content
!git clone https://github.com/krishhhna7/rag-poison-detect.git || (cd rag-poison-detect && git pull)
%cd /content/rag-poison-detect
!pip -q install transformers accelerate pyyaml tqdm

In [ ]:
!python -m pytest -q

## Step 3. Check what is saved on Drive
You should see both run folders, each with `features.csv`, `results_summary.csv`, `significance.csv`, `oof_flags.csv`, `filter_results.jsonl`
(and `attack_summary.csv` for both). If something is missing, tell me which file before continuing.

In [ ]:
!ls -la $RAGDET_ROOT/runs/nq_qwen3b_contriever
!echo
!ls -la $RAGDET_ROOT/runs/nq_qwen3b_n1

## Step 4. Build the report tables and the results bundle (CPU, about 2 minutes)
**4a. Regenerate every table** into one file, `analysis_outputs.txt` (also saved on Drive). Read it to see all the final numbers.

In [ ]:
!python -m ragdet.cli report --config configs/small_gpu.yaml

**4b. Download everything for GitHub.** This copies only the small result files into one zip and downloads it to your laptop.
Unzip it and copy its contents into the `results` folder of your project (see the instructions I gave you), then `git add`, `git commit`, `git push`.

In [ ]:
import os, shutil
from google.colab import files
ROOT = os.environ["RAGDET_ROOT"]
RUNS = {"main_3poison": "nq_qwen3b_contriever", "ablation_1poison": "nq_qwen3b_n1"}
KEEP = ["attack_summary.csv", "features.csv", "results_summary.csv", "results_raw.csv",
        "significance.csv", "filter_results.jsonl", "oof_flags.csv"]
out = f"{ROOT}/results_bundle"
shutil.rmtree(out, ignore_errors=True)
for name, run in RUNS.items():
    os.makedirs(f"{out}/{name}", exist_ok=True)
    for f in KEEP:
        src = f"{ROOT}/runs/{run}/{f}"
        if os.path.exists(src):
            shutil.copy(src, f"{out}/{name}/{f}")
        else:
            print("MISSING:", run, f)
shutil.copy(f"{ROOT}/analysis_outputs.txt", f"{out}/analysis_outputs.txt")
zip_path = shutil.make_archive(f"{ROOT}/results_bundle", "zip", out)
print("zip:", zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")
files.download(zip_path)

## Step 5 (OPTIONAL, needs a T4 GPU runtime). Re-check the end-to-end filter results
Everything is already saved, so these finish in about 2 minutes (they print `cached`). Only run them if Step 3 showed `filter_results.jsonl` missing.
If you run them, switch to a GPU runtime, re-run Steps 1 and 2, then run Step 4a again to refresh the tables.

In [ ]:
!python -m ragdet.cli filter --config configs/small_gpu.yaml --set "eval.filter_detectors=['fusion','sem','internal','ppl_filter','fusion_minus_ppl']"

In [ ]:
!python -m ragdet.cli filter --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=nq_qwen3b_n1 "eval.filter_detectors=['fusion','sem','internal','ppl_filter','fusion_minus_ppl']"

## Optional: refresh the detection tables (CPU, about 5 minutes each)
Not needed unless you changed something. Same numbers come out (everything is seeded).

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=nq_qwen3b_n1

---
# APPENDIX (skip unless rebuilding from scratch; about 2.5 hours of T4 GPU)
Use NEW run names so nothing already saved is touched. Needs the sub-corpus and embeddings already in `MyDrive/ragdet/cache` (they are).
Run each stage only after the previous one finished. If Colab disconnects: re-run Steps 1 and 2, then re-run the SAME stage (attack and a2 resume).
Time on a T4: attack 16 min, features 13 min, a2 33 min, evaluate 5 min, filter 20 min.

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml --set run_name=rebuild_n3

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml --set run_name=rebuild_n3

In [ ]:
!python -m ragdet.cli a2       --config configs/small_gpu.yaml --set run_name=rebuild_n3

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml --set run_name=rebuild_n3

In [ ]:
!python -m ragdet.cli filter   --config configs/small_gpu.yaml --set run_name=rebuild_n3 "eval.filter_detectors=['fusion','sem','internal','ppl_filter','fusion_minus_ppl']"

1-poison rebuild (no A2):

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=rebuild_n1

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=rebuild_n1

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=rebuild_n1

In [ ]:
!python -m ragdet.cli filter   --config configs/small_gpu.yaml --set attack.n_poison=1 run_name=rebuild_n1 "eval.filter_detectors=['fusion','sem','internal','ppl_filter','fusion_minus_ppl']"